# Experiment 10
## Analysis of GAN Loss Functions
**Aim:** Analyze and compare different GAN loss functions — Original (Minimax), Least Squares, and Wasserstein.

### Step 1 – Import Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

### Step 2 – Simulate Discriminator Output Scores
We simulate discriminator outputs for real and fake images for analysis.

In [ ]:
torch.manual_seed(42)
D_real = torch.rand(100) * 0.4 + 0.6    # D(x) → scores near 1 for real
D_fake = torch.rand(100) * 0.4 + 0.0    # D(G(z)) → scores near 0 for fake
print(f'D(real) mean: {D_real.mean():.3f} | D(fake) mean: {D_fake.mean():.3f}')

### Step 3 – Original Minimax GAN Loss (Binary Cross-Entropy)

In [ ]:
bce = nn.BCELoss()

# Discriminator wants to maximize (real→1, fake→0)
d_loss_orig = bce(D_real, torch.ones_like(D_real)) + bce(D_fake, torch.zeros_like(D_fake))

# Generator wants fake to look real
g_loss_orig = bce(D_fake, torch.ones_like(D_fake))

print(f'[Minimax GAN]  D Loss: {d_loss_orig.item():.4f} | G Loss: {g_loss_orig.item():.4f}')

### Step 4 – Least Squares GAN (LSGAN)
LSGAN uses mean squared error instead of cross-entropy, reducing vanishing gradients.

In [ ]:
# D wants D(real)=1, D(fake)=0
d_loss_ls = 0.5 * torch.mean((D_real - 1)**2) + 0.5 * torch.mean(D_fake**2)

# G wants D(fake)=1
g_loss_ls = 0.5 * torch.mean((D_fake - 1)**2)

print(f'[LSGAN]        D Loss: {d_loss_ls.item():.4f} | G Loss: {g_loss_ls.item():.4f}')

### Step 5 – Wasserstein GAN (WGAN) Loss
WGAN uses Earth Mover's Distance — more stable training and meaningful metric.

In [ ]:
# D wants to maximize D(real) - D(fake)  →  minimize -(D(real) - D(fake))
D_real_w = D_real - 0.5    # critic outputs (can be any real number, not bounded to [0,1])
D_fake_w = D_fake - 0.5

d_loss_w = -torch.mean(D_real_w) + torch.mean(D_fake_w)   # minimize negative EM distance
g_loss_w = -torch.mean(D_fake_w)                           # generator maximizes D(fake)

print(f'[WGAN]         D Loss: {d_loss_w.item():.4f} | G Loss: {g_loss_w.item():.4f}')

### Step 6 – Compare Losses Visually

In [ ]:
losses = {'Minimax GAN': (d_loss_orig.item(), g_loss_orig.item()),
          'LSGAN':       (d_loss_ls.item(),   g_loss_ls.item()),
          'WGAN':        (abs(d_loss_w.item()), abs(g_loss_w.item()))}

names    = list(losses.keys())
d_vals   = [v[0] for v in losses.values()]
g_vals   = [v[1] for v in losses.values()]
x        = np.arange(len(names))

plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, d_vals, width=0.35, label='Discriminator Loss', color='steelblue')
plt.bar(x + 0.2, g_vals, width=0.35, label='Generator Loss',     color='orange')
plt.xticks(x, names); plt.ylabel('Loss Value')
plt.title('Comparison of GAN Loss Functions'); plt.legend(); plt.grid(axis='y'); plt.show()

### Step 7 – Feature Matching Loss
Feature matching stabilizes GAN training by matching intermediate feature statistics.

In [ ]:
# Simulate feature activations from D's hidden layer
real_features = torch.randn(100, 64)
fake_features = torch.randn(100, 64)

feature_match_loss = torch.mean((real_features.mean(0) - fake_features.mean(0))**2)
print(f'Feature Matching Loss: {feature_match_loss.item():.4f}')

### Result
| Loss Type | Discriminator | Generator | Notes |
|---|---|---|---|
| Minimax GAN | BCE | BCE | Can suffer from vanishing gradients |
| LSGAN | MSE | MSE | Smoother gradients, more stable |
| WGAN | Earth Mover | Earth Mover | Most stable, meaningful metric |

All three GAN loss functions were implemented and compared. WGAN provides the most meaningful training signal.